# 04 · Situaciones inesperadas

**Pregunta central 3 del Plan Maestro:** ¿cómo reacciona el equipo ante situaciones inesperadas (gol en contra, expulsión, ir perdiendo, etc.)?

Este bloque define primero, con criterio explícito, qué cuenta como "evento disparador" (ver `identificar_eventos_disparadores`), y luego compara el comportamiento del equipo en los minutos posteriores contra su "promedio normal" (línea base) a lo largo de todo el dataset.

Los 4 tipos de evento disparador considerados:
- **gol_recibido** / **gol_recibido_temprano** (antes del min 20): el rival anota.
- **expulsion_propia** / **expulsion_rival**: tarjeta roja o segunda amarilla a cualquiera de los dos equipos.
- **perdiendo_al_descanso**: el equipo se fue abajo en el marcador al finalizar el primer tiempo (el evento se ancla al arranque del segundo tiempo).

In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from framework_analitico import (
    normalizar_direccion_ataque,
    identificar_goles,
    identificar_expulsiones,
    marcador_al_descanso,
    identificar_eventos_disparadores,
    metricas_ventana,
    linea_base_equipo,
    comparar_metricas_pre_post_evento,
)

EQUIPO = "América"  # cambia esto para reaplicar el analisis a otro entrenador/equipo

events_jardine = pd.read_csv("../data/raw/events_jardine_america.csv", low_memory=False)
print(f"Eventos cargados: {len(events_jardine)}")

# Normalizar dirección de ataque ANTES de todo lo demás (altura_promedio_x usa x normalizada)
events_norm = normalizar_direccion_ataque(events_jardine, equipo=EQUIPO)

print(f"Partidos: {events_norm['match_id'].nunique()}")

## 1. Eventos disparadores identificados

Catálogo completo de situaciones inesperadas detectadas en los 129 partidos, con criterio explícito (ver docstring de `identificar_eventos_disparadores`).

In [ ]:
disparadores = identificar_eventos_disparadores(events_norm, equipo=EQUIPO)

print(f"Total de eventos disparadores: {len(disparadores)}")
print()
print(disparadores["tipo_disparador"].value_counts())
print()
disparadores.head(10)

## 2. Línea base del equipo

El "promedio normal" de `América` a lo largo de todo el dataset, expresado en tasas (%, por minuto) para que sea comparable contra ventanas cortas post-evento.

In [ ]:
base = linea_base_equipo(events_norm, equipo=EQUIPO)
base

## 3. Comparación: métricas en los 5 minutos posteriores al evento vs. línea base

Para cada evento disparador, se calculan las métricas de `metricas_ventana` en los 300 segundos (5 min) posteriores, y se comparan contra la línea base. Las columnas `diff_*` resumen la diferencia (post - base): positivo = el equipo *sube* esa métrica tras el evento; negativo = se *repliega*.

In [ ]:
comparacion = comparar_metricas_pre_post_evento(
    events_norm, equipo=EQUIPO, eventos_disparadores=disparadores, ventana_segundos=300
)

comparacion

## 4. Promedio de la reacción por tipo de evento disparador

Se agrupan los `diff_*` por tipo de evento disparador para ver el patrón general de reacción (no solo caso por caso).

In [ ]:
columnas_diff = [c for c in comparacion.columns if c.startswith("diff_")]
resumen_por_tipo = comparacion.groupby("tipo_disparador")[columnas_diff].mean().round(3)
resumen_por_tipo["n_casos"] = comparacion.groupby("tipo_disparador").size()

resumen_por_tipo

## 5. Visualización: reacción de presión e intensidad ofensiva por tipo de evento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tipos = resumen_por_tipo.index

axes[0].barh(tipos, resumen_por_tipo["diff_presiones_por_min"], color="#8B0000")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Cambio en presiones/min tras el evento\n(vs. línea base)")
axes[0].set_xlabel("diff presiones_por_min")

axes[1].barh(tipos, resumen_por_tipo["diff_posesion_pct"], color="#FFD700", edgecolor="black")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Cambio en % de posesión tras el evento\n(vs. línea base)")
axes[1].set_xlabel("diff posesion_pct (puntos porcentuales)")

plt.tight_layout()
plt.savefig("../outputs/figures/reaccion_eventos_disparadores.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Caso particular: marcador al descanso

Detalle de en cuántos partidos el equipo se fue ganando, empatando o perdiendo al descanso, como contexto adicional al evento `perdiendo_al_descanso`.

In [ ]:
marcador = marcador_al_descanso(events_norm, equipo=EQUIPO)

print(marcador["situacion_al_descanso"].value_counts())
print()
marcador[marcador["situacion_al_descanso"] == "perdiendo"]

## 7. Síntesis: ¿cómo reacciona el equipo ante situaciones inesperadas?

Se identificaron **204 eventos disparadores** en los 129 partidos (gol_recibido: 120, expulsion_rival: 27, gol_recibido_temprano: 25, perdiendo_al_descanso: 23, expulsion_propia: 9), y se comparó el comportamiento de América en los 5 minutos posteriores a cada uno contra su línea base de todo el dataset (posesión 54.7%, 41.2% pases largos, 0.98 presiones/min, altura promedio x=59.4, 0.093 tiros/min a favor).

**La reacción NO es uniforme: depende mucho de qué tipo de disparador sea.**

- **Expulsión propia (repliegue de manual):** caída fuerte en posesión (-13.4pp) y en altura de línea (-10.3, la mayor de la tabla), sube mucho el % de pases largos (+26.5pp, alivia presión jugando directo) y bajan los tiros a favor. El equipo se compacta y juega más directo cuando se queda en inferioridad numérica.
- **Expulsión rival (controla, no acelera):** la posesión y la altura suben con fuerza (+12.8pp / +11.9), pero la presión **cae** (-0.68/min, la mayor caída de toda la tabla). Con el hombre de más, América controla el partido con el balón en vez de presionar con más intensidad — maneja el ritmo en lugar de ahogar al rival.
- **Gol recibido (caso general):** reacción tibia — sube poco la posesión (+3.5pp) y casi nada la presión (+0.07/min); la altura incluso baja (-3.6). Un gol encajado cualquiera no dispara, por sí solo, una respuesta ofensiva fuerte.
- **Gol recibido temprano (<min 20):** la presión sube casi 5 veces más que en el caso general (+0.33 vs +0.07/min). Con tiempo de sobra en el partido, Jardine empuja con bastante más intensidad defensiva para recuperar rápido.
- **Perdiendo al descanso (la reacción más ofensiva de las 5):** es el único disparador donde suben a la vez posesión (+9.4pp), presión (+0.41/min) y altura (+2.6), y además es el que más tiros genera (+0.124/min, ~2x la tasa base). El equipo sale claramente a buscar el partido en el arranque del segundo tiempo. Contexto: esto ocurrió en 23 de 129 partidos (17.8%) — poco frecuente, pero con una respuesta muy nítida cuando pasa.

**Lectura general para el Plan Maestro:** Jardine no tiene una sola "respuesta automática" ante lo inesperado — reacciona distinto según el tipo de contratiempo. Ante la adversidad vinculada al marcador (ir perdiendo, sobre todo al descanso) responde con más intensidad y más llegada; ante la adversidad numérica (expulsión propia) se repliega y simplifica el juego; y ante una ventaja numérica (expulsión rival) prioriza el control del balón por encima de la presión. El patrón más "defensivo puro" es la expulsión propia, y el más "ofensivo puro" es ir perdiendo al descanso.